# Inference Playground (NLP + Sugar Regression)

This notebook provides a minimal inference workflow with optional NLP preprocessing using `nltk`, `spaCy`, and `transformers` tokenizers.

In [ ]:
import re
import unicodedata
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

def clean_ingredients(text: str) -> str:
    text = unicodedata.normalize('NFKC', text or '')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

model_id = 'ziq/ingbetic'
tokenizer_id = 'distilbert-base-uncased-finetuned-sst-2-english'

tokenizer = AutoTokenizer.from_pretrained(tokenizer_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id)
model.eval()

In [ ]:
sample = "4 corn tortillas, 1/2 teaspoon oregano, 3/4 cup shredded jack cheese, Salt and pepper"
cleaned = clean_ingredients(sample)
enc = tokenizer(cleaned, truncation=True, padding='max_length', max_length=128, return_tensors='pt')

with torch.no_grad():
    pred_z = model(**enc).logits.squeeze().item()

# Convert normalized output to grams if using project-specific scaling
pred_sugar_g = pred_z * 13.3627190349059 + 10.85810766787474
print({'cleaned': cleaned, 'pred_sugar_z': pred_z, 'pred_sugar_g': pred_sugar_g})